In [1]:
import glob
import pandas as pd

In [2]:
combined_df = pd.DataFrame()

# Use correct path relative to notebook location (in src/)
for file in glob.glob("../results/results_*.csv"):
    df = pd.read_csv(file)
    combined_df = pd.concat([combined_df, df], ignore_index=True)

print(f"Loaded {len(combined_df)} rows from results files")

Loaded 390 rows from results files


In [ ]:
combined_df.head()

In [ ]:
# Check what columns are in the dataframe
print("Columns in combined_df:")
print(combined_df.columns.tolist())
print("\nFirst few rows:")
print(combined_df.head())

In [ ]:
# Check what models we have
print("Unique models:", sorted(combined_df['model'].unique()))

In [ ]:
import math
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for dev container
import matplotlib.pyplot as plt
import seaborn as sns

# Set publication-ready style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],  # Available on Ubuntu by default
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 100,  # Lower for notebook display
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.5,
    'lines.linewidth': 2,
    'lines.markersize': 6,
})

def plot_species_models_publication(df, output_path="results/species_comparison.png", species_list=None):
    """Create publication-quality plots with 95% confidence intervals."""
    
    # Color palette for models
    colors = {
        'birdnet': '#2E86AB',
        'mobilenet': '#A23B72',
        'perch': '#F18F01',
        'resnet': '#06A77D',
        'vgg': '#C73E1D',
    }
    
    if species_list is None:
        species_list = sorted(df["species"].unique())
    
    n_species = len(species_list)
    n_cols = 2
    n_rows = math.ceil(n_species / n_cols)
    
    # sharex/sharey hides redundant inner tick labels automatically
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True)
    axes = axes.flatten()
    
    # Hide unused axes
    for j in range(n_species, len(axes)):
        axes[j].set_visible(False)

    # Store handles and labels for shared legend
    handles, labels = None, None

    for i, species in enumerate(species_list):
        ax = axes[i]
        species_data = df[df["species"] == species]
        models = sorted(species_data["model"].unique())
        
        for model in models:
            model_data = species_data[species_data["model"] == model].sort_values("training_size")
            
            ax.errorbar(
                model_data["training_size"],
                model_data["test_auc_mean"],
                yerr=model_data["test_auc_ci_95"],
                label=model.upper(),
                marker='o',
                color=colors.get(model, '#333333'),
                capsize=4,
                capthick=1.5,
                elinewidth=1.5,
                alpha=0.9,
                markeredgewidth=0.5,
                markeredgecolor='white',
            )
        
        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
        
        species_name = species.replace('_', ' ').title()
        ax.set_title(species_name, fontweight='semibold', pad=10)
        
        ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)
        ax.set_axisbelow(True)
        ax.set_ylim(0.2, 1.02)
        ax.set_xlim(-5, 175)
        ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5, zorder=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # Single shared axis labels for the whole figure
    fig.supxlabel("Training Size (samples per class)", fontweight='medium', fontsize=12, y=0.13)
    fig.supylabel("Test ROC-AUC", fontweight='medium', fontsize=12, x=0.02)

    # Legend below all subplots
    fig.legend(handles, labels,
               loc='lower center',
               ncol=len(labels) if labels else 5,
               frameon=True,
               fancybox=False,
               edgecolor='gray',
               framealpha=0.95,
               fontsize=11,
               bbox_to_anchor=(0.5, 0.04))
    
    plt.subplots_adjust(hspace=0.2, wspace=0.1, bottom=0.22)
    
    plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.3, facecolor='white')
    print(f"✓ Plot saved to {output_path}")
    plt.close()
    return fig

In [ ]:
plot_species_models_publication(
    combined_df,
    "../_anthro_comparison.png",
    species_list=[
    "engine",
    "traffic",
    "generator",
    "power_tools",
    "device_static",
    "wind"
    ]
)

In [3]:
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Species ordered by ecological category ────────────────────────────────────
GROUPS = [
    ("Amphibians",      ["american_bullfrog", "pacific_chorus_frog",
                         "woodhouses_toad",   "yellow_legged_frog"]),
    ("Mammals",         ["coyote", "nutria"]),
    ("Insects & Human", ["field_cricket", "human_vocal"]),
    ("Anthropogenic",   ["engine", "generator", "power_tools",
                         "traffic", "device_static", "wind"]),
]

MODEL_COLORS = {
    'birdnet':   '#2E86AB',
    'mobilenet': '#A23B72',
    'perch':     '#F18F01',
    'resnet':    '#06A77D',
    'vgg':       '#C73E1D',
}
CAT_BG = {
    "Amphibians":      "#EEF6FB",
    "Mammals":         "#EEFBEE",
    "Insects & Human": "#FFFBEE",
    "Anthropogenic":   "#F8F0FB",
}

species_order, species_cat, cat_ranges = [], {}, {}
for cat, slist in GROUPS:
    start = len(species_order)
    for s in slist:
        species_order.append(s)
        species_cat[s] = cat
    cat_ranges[cat] = (start, len(species_order) - 1)

n_species = len(species_order)   # 14
n_cols    = 2
n_rows    = n_species // n_cols  # 7

_df = combined_df.copy()

# ── Figure: left = training curves (7×2), right = heatmap ────────────────────
fig = plt.figure(figsize=(18, 26))
outer = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[2.2, 1],
                          left=0.08, right=0.97, wspace=0.18,
                          bottom=0.07, top=0.97)

left_gs = gridspec.GridSpecFromSubplotSpec(
    n_rows, n_cols, subplot_spec=outer[0], hspace=0.38, wspace=0.08)

curve_axes = np.array([[fig.add_subplot(left_gs[r, c]) for c in range(n_cols)]
                        for r in range(n_rows)]).flatten()

# Share axes; hide inner tick labels
for ax in curve_axes[1:]:
    ax.sharex(curve_axes[0])
    ax.sharey(curve_axes[0])
    plt.setp(ax.get_xticklabels(), visible=False)
    plt.setp(ax.get_yticklabels(), visible=False)
plt.setp(curve_axes[0].get_xticklabels(), visible=False)
for ax in curve_axes[-n_cols:]:                        # bottom row: show x ticks
    plt.setp(ax.get_xticklabels(), visible=True)
for r in range(n_rows):                                # left col: show y ticks
    plt.setp(curve_axes[r * n_cols].get_yticklabels(), visible=True)

handles_leg, labels_leg = None, None

for i, species in enumerate(species_order):
    ax = curve_axes[i]
    ax.set_facecolor(CAT_BG[species_cat[species]])

    sp_data = _df[_df["species"] == species]
    for model in sorted(sp_data["model"].unique()):
        md = sp_data[sp_data["model"] == model].sort_values("training_size")
        ax.errorbar(
            md["training_size"], md["test_auc_mean"],
            yerr=md["test_auc_ci_95"],
            label=model.upper(),
            marker='o', markersize=4.5,
            color=MODEL_COLORS.get(model, '#333'),
            capsize=3, capthick=1.1, elinewidth=1.1,
            linewidth=1.8, alpha=0.9,
            markeredgewidth=0.4, markeredgecolor='white',
        )

    if handles_leg is None:
        handles_leg, labels_leg = ax.get_legend_handles_labels()

    ax.set_title(species.replace('_', ' ').title(),
                 fontweight='semibold', pad=6, fontsize=10)
    ax.set_ylim(0.2, 1.02)
    ax.set_xlim(-5, 175)
    ax.axhline(0.5, color='#aaa', linestyle=':', linewidth=0.8, zorder=0)
    ax.grid(True, alpha=0.22, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=8)

# ── Category separators and margin labels (after layout is finalised) ─────────
fig.canvas.draw()

for cat, (start, end) in cat_ranges.items():
    pos_first = curve_axes[start].get_position()
    pos_last  = curve_axes[end].get_position()
    y_mid = (pos_first.y1 + pos_last.y0) / 2

    fig.text(0.005, y_mid, cat, rotation=90, va='center', ha='center',
             fontsize=9, fontweight='bold', color='#444',
             transform=fig.transFigure)

    if start > 0:
        prev_left_idx = (start - 1) - ((start - 1) % n_cols)
        pos_prev = curve_axes[prev_left_idx].get_position()
        y_sep = (pos_prev.y0 + pos_first.y1) / 2
        right_x = curve_axes[min(start + 1, n_species - 1)].get_position().x1
        fig.add_artist(plt.Line2D(
            [pos_first.x0, right_x], [y_sep, y_sep],
            transform=fig.transFigure,
            color='#bbbbbb', linewidth=0.9, linestyle='--', zorder=10,
        ))

# Shared axis labels for the left panel
left_pos = outer[0].get_position(fig)
fig.text(left_pos.x0 + (left_pos.x1 - left_pos.x0) / 2, 0.025,
         "Training Size (samples per class)",
         ha='center', fontsize=11, fontweight='medium')
fig.text(0.03, left_pos.y0 + (left_pos.y1 - left_pos.y0) / 2,
         "Test ROC-AUC", rotation=90, va='center',
         fontsize=11, fontweight='medium')

# ── Right panel: heatmap of AUC at largest training size ─────────────────────
max_n = _df["training_size"].max()
heat_data = (
    _df[_df["training_size"] == max_n]
    .groupby(["species", "model"])["test_auc_mean"]
    .mean()
    .unstack("model")
    .reindex(species_order)
)
model_order = sorted(heat_data.columns)
heat_data = heat_data[model_order]

ax_heat = fig.add_subplot(outer[1])
im = ax_heat.imshow(heat_data.values, aspect='auto', cmap='RdYlGn',
                    vmin=0.5, vmax=1.0, interpolation='nearest')

ax_heat.set_xticks(np.arange(len(model_order)))
ax_heat.set_xticklabels([m.upper() for m in model_order],
                         rotation=35, ha='right', fontsize=9)
ax_heat.set_yticks(np.arange(n_species))
ax_heat.set_yticklabels([s.replace('_', ' ').title() for s in species_order],
                         fontsize=9)
ax_heat.tick_params(length=0)

for r in range(n_species):
    for c in range(len(model_order)):
        val = heat_data.values[r, c]
        if not np.isnan(val):
            txt_color = 'white' if val < 0.65 or val > 0.90 else '#222'
            ax_heat.text(c, r, f'{val:.2f}', ha='center', va='center',
                         fontsize=7.5, color=txt_color, fontweight='medium')

# Category separators and background bands on heatmap
for cat, (start, end) in cat_ranges.items():
    rgba = plt.matplotlib.colors.to_rgba(CAT_BG[cat], alpha=0.35)
    ax_heat.add_patch(plt.Rectangle(
        (-0.5, start - 0.5), len(model_order), end - start + 1,
        color=rgba, zorder=0))
    if start > 0:
        ax_heat.axhline(start - 0.5, color='white', linewidth=2.5)
        ax_heat.axhline(start - 0.5, color='#888', linewidth=0.7, linestyle='--')

ax_heat.set_title(f"AUC at n={int(max_n)}\n(largest training set)",
                  fontsize=10, fontweight='semibold', pad=10)
ax_heat.set_xlabel("Model", fontsize=10, fontweight='medium')
ax_heat.spines[:].set_visible(False)

cbar = fig.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04,
                    ticks=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
cbar.ax.tick_params(labelsize=8)
cbar.set_label("AUC", fontsize=9)

# ── Shared legend at bottom ───────────────────────────────────────────────────
fig.legend(handles_leg, labels_leg,
           loc='lower center', ncol=len(labels_leg),
           frameon=True, fancybox=False,
           edgecolor='#ccc', framealpha=0.95, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))

plt.savefig("../_all_species_overview.png", dpi=300, bbox_inches='tight',
            pad_inches=0.3, facecolor='white')
print("✓ Saved: ../_all_species_overview.png")
plt.close()

✓ Saved: ../_all_species_overview.png
